In [ ]:
# ============================================================
# COLAB TRANSFER - 03: UPLOAD PARA KAGGLE DATASET (CLI)
# ============================================================
# Este notebook roda no GOOGLE COLAB
# Objetivo: /content/kaggle_staging → automamermaid/comfydocs
# Usa script compartilhado: scripts/kaggle_upload.py
# ============================================================

import sys
import json
import os
from pathlib import Path

# Adicionar scripts ao path
sys.path.insert(0, "/content/scripts")

# --- Garantir autenticação Kaggle (lê de Secrets do Colab via userdata) ---
def ensure_kaggle_auth():
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_json = kaggle_dir / "kaggle.json"
    
    if kaggle_json.exists():
        return True
    
    username = None
    key = None
    try:
        from google.colab import userdata
        username = userdata.get('KAGGLE_USERNAME')
        key = userdata.get('KAGGLE_KEY')
    except (ImportError, userdata.NotebookAccessError):
        pass
    
    if not username:
        username = os.environ.get("KAGGLE_USERNAME")
    if not key:
        key = os.environ.get("KAGGLE_KEY")
    
    if username and key:
        kaggle_dir.mkdir(exist_ok=True)
        kaggle_json.write_text(json.dumps({"username": username, "key": key}))
        os.chmod(kaggle_json, 0o600)
        print(f"Kaggle auth: CRIADO a partir de Secrets do Colab ({username})")
        return True
    
    raise RuntimeError("KAGGLE_USERNAME e KAGGLE_KEY não configurados nos Secrets do Colab")

ensure_kaggle_auth()

print("=" * 60)
print("UPLOAD STAGING → KAGGLE DATASET (CLI)")
print("=" * 60)

# Configurações
MODEL_NAME = "lustifyNSFWCheckpoint_v10Krea2.safetensors"
DATASET = "automamermaid/comfydocs"
STAGING_DIR = Path("/content/kaggle_staging")
VERSION_NOTES = "Add lustifyNSFWCheckpoint_v10Krea2 FP8 ~12GB"

print(f"Modelo: {MODEL_NAME}")
print(f"Dataset: {DATASET}")
print(f"Staging: {STAGING_DIR}")

# Verificar se modelo existe no staging
model_path = STAGING_DIR / MODEL_NAME
if not model_path.exists():
    raise FileNotFoundError(f"Modelo não encontrado em {model_path}. Execute 02_download.ipynb primeiro.")

size = model_path.stat().st_size
size_gb = size / (1024**3)
print(f"Tamanho: {size:,} bytes ({size_gb:.2f} GB)")

if size == 0:
    raise ValueError("Arquivo tem 0 bytes, não será enviado")

# Executar upload via script compartilhado
from kaggle_upload import upload_model

try:
    upload_model(
        model_name=MODEL_NAME,
        staging_dir=STAGING_DIR,
        dataset=DATASET,
        method="cli",
        version_notes=VERSION_NOTES,
    )
    print(f"\n✅ SUCESSO: Upload concluído para {DATASET}")
except PermissionError as e:
    print(f"\n⚠️  CLI retornou 403: {e}")
    print("Tentando fallback via kagglehub...")
    try:
        upload_model(
            model_name=MODEL_NAME,
            staging_dir=STAGING_DIR,
            dataset=DATASET,
            method="kagglehub",
            version_notes=VERSION_NOTES,
        )
        print(f"\n✅ SUCESSO: Upload via kagglehub concluído")
    except Exception as e2:
        print(f"\n❌ kagglehub também falhou: {e2}")
        raise RuntimeError("Ambos métodos de upload falharam")
except Exception as e:
    print(f"\n❌ FALHA: {e}")
    import traceback
    traceback.print_exc()
    raise

print("\n" + "=" * 60)
print("UPLOAD CONCLUÍDO - MODELO DISPONÍVEL NO KAGGLE DATASET")
print("=" * 60)
print("\nPRÓXIMO PASSO (no Kaggle Notebook):")
print("  Execute kaggle_runtime/05_download_to_local.ipynb")
print("  ou kaggle_runtime/08_master_pipeline.ipynb")